In [ ]:
%%sql -r dataframe_5
use database snowflake_learning_db;

In [ ]:
%%sql -r dataframe_1
show accounts

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
import pandas as pd

df_sp = session.sql("SHOW ACCOUNTS")
df = pd.DataFrame(df_sp.collect())
print(df)

In [ ]:
# Let's assume you've already renamed the columns and added snowsight_url
df["snowsight_url"] = (
    "https://app.snowflake.com/"
    + df["organization_name"].str.lower()
    + "/"
    + df["account_name"].str.lower()
)

# Define fields you want to show
fields = [
    "organization_name",
    "account_name",
    "account_locator",
    "account_url",
    "account_locator_url",
    "snowsight_url"
]

# Loop through each account
for i, row in df.iterrows():
    for field in fields:
        print(f"{field:22}: {row[field]}")

In [ ]:
snow_df = session.create_dataframe(df)

snow_df.write.mode("overwrite").save_as_table("TEMP_ACCOUNTS", table_type="temporary")

In [ ]:
%%sql -r dataframe_2
SELECT * FROM TEMP_ACCOUNTS;

In [ ]:
%%sql -r dataframe_3
SELECT * 
FROM (
  SELECT "organization_name", "account_name", "account_locator", "account_url", "account_locator_url", 
         'https://app.snowflake.com/' || LOWER("organization_name") || '/' || LOWER("account_name") AS "snowsight_url"
  FROM TEMP_ACCOUNTS
)
UNPIVOT (
  value FOR key IN (
    "organization_name",
    "account_name",
    "account_locator",
    "account_url",
    "account_locator_url",
    "snowsight_url"
  )
);